<a href="https://www.kaggle.com/code/izzarsulynashrudin/pcqm4mv2?scriptVersionId=308970694" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Baseline PCQM4Mv2 untuk Kaggle

Notebook ini menyesuaikan pipeline lokal ke environment Kaggle dengan sumber data resmi `PCQM4Mv2` dari package `ogb`, jadi tidak lagi membaca file SDF lokal.

Target eksperimennya mengikuti dokumentasi resmi OGB, yaitu memprediksi `HOMO-LUMO gap` dari representasi molekul berbasis `SMILES` dan split resmi `train/valid/test-dev/test-challenge`.


## Import dan Setup Kaggle

Bagian ini memasang dependency yang dibutuhkan di Kaggle, memuat library analisis, lalu menyiapkan helper model dan utilitas umum untuk seluruh notebook.


In [ ]:
import importlib
import json
import math
import os
import pickle
import shutil
import subprocess
import sys
import time
import warnings
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")


def ensure_package(import_name, install_name=None):
    try:
        return importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", install_name or import_name]
        )
        return importlib.import_module(import_name)


ensure_package("ogb", "ogb")
try:
    from rdkit import Chem
    from rdkit.Chem import AllChem
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rdkit-pypi"])
    from rdkit import Chem
    from rdkit.Chem import AllChem

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from ogb.lsc import PCQM4Mv2Dataset, PCQM4Mv2Evaluator
from sklearn.base import clone
from sklearn.ensemble import (
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    explained_variance_score,
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    from xgboost import XGBRegressor
    xgboostAvailable = True
except ImportError:
    XGBRegressor = None
    xgboostAvailable = False


DEFAULT_FEATURE_COLUMNS = ['atomCount', 'heavyAtomCount', 'hydrogenCount', 'bondCount', 'singleBondCount', 'doubleBondCount', 'tripleBondCount', 'aromaticBondCount', 'uniqueElementCount', 'heteroAtomCount', 'heteroAtomFraction', 'totalAtomicNumber', 'meanAtomicNumber', 'totalAtomicMass', 'meanAtomicMass', 'degreeMean', 'degreeStd', 'degreeMax', 'degreeMin', 'terminalAtomFraction', 'branchAtomFraction', 'bondToAtomRatio', 'ringIndex', 'elementCountC', 'elementCountH', 'elementCountN', 'elementCountO', 'elementCountF', 'elementCountP', 'elementCountS', 'elementCountCl', 'elementCountBr', 'elementCountI']

DEFAULT_ELEMENT_COLORS = {
    "H": "#94a3b8",
    "C": "#cbd5e1",
    "N": "#60a5fa",
    "O": "#f87171",
    "F": "#34d399",
    "P": "#f59e0b",
    "S": "#c084fc",
    "Cl": "#4ade80",
    "Br": "#fb923c",
    "I": "#a78bfa",
    "B": "#fbbf24",
}

DEFAULT_ELEMENT_SIZES = {
    "H": 0.9,
    "C": 1.6,
    "N": 1.7,
    "O": 1.7,
    "F": 1.65,
    "P": 1.9,
    "S": 1.9,
    "Cl": 2.0,
    "Br": 2.0,
    "I": 2.2,
}


def build_model_map(random_state):
    model_map = {
        "ridgeRegression": Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                ("ridge", Ridge(alpha=1.0)),
            ]
        ),
        "randomForest": RandomForestRegressor(
            n_estimators=250,
            max_depth=None,
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=random_state,
        ),
        "extraTrees": ExtraTreesRegressor(
            n_estimators=320,
            max_depth=None,
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=random_state,
        ),
        "gradientBoosting": GradientBoostingRegressor(
            learning_rate=0.05,
            n_estimators=280,
            max_depth=5,
            min_samples_leaf=8,
            random_state=random_state,
        ),
        "histGradientBoosting": HistGradientBoostingRegressor(
            learning_rate=0.06,
            max_depth=10,
            max_iter=300,
            min_samples_leaf=20,
            random_state=random_state,
        ),
    }

    if XGBRegressor is not None:
        model_map["xgboost"] = XGBRegressor(
            n_estimators=420,
            max_depth=8,
            learning_rate=0.05,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.0,
            reg_lambda=1.0,
            objective="reg:squarederror",
            tree_method="hist",
            n_jobs=-1,
            random_state=random_state,
        )

    return model_map


def build_importance_frame(model_object, feature_column_names):
    if hasattr(model_object, "feature_importances_"):
        importance_series = pd.Series(model_object.feature_importances_, index=feature_column_names)
    elif hasattr(model_object, "named_steps") and "ridge" in model_object.named_steps:
        importance_series = pd.Series(
            np.abs(model_object.named_steps["ridge"].coef_),
            index=feature_column_names,
        )
    else:
        return pd.DataFrame(columns=["featureName", "importance"])

    importance_frame = (
        importance_series.sort_values(ascending=False)
        .head(12)
        .rename("importance")
        .reset_index()
    )
    importance_frame.columns = ["featureName", "importance"]
    return importance_frame


warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")
sns.set_theme(style="whitegrid", palette="crest")
plt.rcParams["figure.figsize"] = (10, 5)

print("Environment Kaggle siap dipakai.")
print(f"XGBoost tersedia: {xgboostAvailable}")


## Ringkasan OGB PCQM4Mv2

Bagian ini mengunduh dataset resmi melalui network menggunakan `PCQM4Mv2Dataset`, mengambil split resmi dari OGB, lalu menetapkan subset eksperimen yang masih realistis untuk sekali run di Kaggle.


In [ ]:
datasetRoot = Path("dataset")
outputDir = Path("output")
figureDir = outputDir / "figures"
viewerDataDir = outputDir / "data"
notebookPath = Path("main.ipynb")

outputDir.mkdir(parents=True, exist_ok=True)
figureDir.mkdir(parents=True, exist_ok=True)
viewerDataDir.mkdir(parents=True, exist_ok=True)

randomState = 42
reportInterval = 5_000
overviewPreviewCount = 3
edaMoleculeSampleCount = 2_000
viewerMoleculeCount = 200
trainSampleLimit = 100_000
validSampleLimit = 10_000
sampleStride = 1
reuseCachedFeatures = True
targetColumnName = "homolumoGap"
officialMetricName = "MAE"
createOfficialTestDevSubmission = False

dataset = PCQM4Mv2Dataset(root=str(datasetRoot), only_smiles=True)
evaluator = PCQM4Mv2Evaluator()
splitDict = dataset.get_idx_split()


def stable_sample(indices, limit, seed):
    indices = np.asarray(indices, dtype=np.int64)
    if limit is None or limit >= len(indices):
        return np.sort(indices.copy())
    rng = np.random.default_rng(seed)
    return np.sort(rng.choice(indices, size=limit, replace=False))


selectedTrainIndices = stable_sample(splitDict["train"], trainSampleLimit, randomState)
selectedValidIndices = stable_sample(splitDict["valid"], validSampleLimit, randomState + 1)

previewRows = []
for previewIndex, datasetIndex in enumerate(selectedTrainIndices[:overviewPreviewCount], start=1):
    smiles, targetValue = dataset[int(datasetIndex)]
    molecule = Chem.AddHs(Chem.MolFromSmiles(smiles))
    previewRows.append(
        {
            "previewIndex": previewIndex,
            "datasetIndex": int(datasetIndex),
            "judulMolekul": smiles[:96] + ("..." if len(smiles) > 96 else ""),
            "atomCount": int(molecule.GetNumAtoms()) if molecule is not None else np.nan,
            "bondCount": int(molecule.GetNumBonds()) if molecule is not None else np.nan,
            targetColumnName: float(targetValue),
        }
    )

totalDatasetCount = len(dataset)
datasetOverviewFrame = pd.DataFrame(
    {
        "parameter": [
            "datasetRoot",
            "datasetObject",
            "targetColumnName",
            "officialMetric",
            "splitTrainFull",
            "splitValidFull",
            "splitTestDevFull",
            "splitTestChallengeFull",
            "trainSampleLimit",
            "validSampleLimit",
            "viewerMoleculeCount",
            "createOfficialTestDevSubmission",
        ],
        "nilai": [
            str(datasetRoot),
            "PCQM4Mv2Dataset(root='dataset', only_smiles=True)",
            targetColumnName,
            officialMetricName,
            int(len(splitDict["train"])),
            int(len(splitDict["valid"])),
            int(len(splitDict["test-dev"])),
            int(len(splitDict["test-challenge"])),
            int(len(selectedTrainIndices)),
            int(len(selectedValidIndices)),
            int(viewerMoleculeCount),
            bool(createOfficialTestDevSubmission),
        ],
    }
)

print(f"Jumlah total molekul pada dataset resmi: {totalDatasetCount:,}")
print("Ringkasan konfigurasi run Kaggle:")
display(datasetOverviewFrame)
print("Cuplikan molekul awal dari split train resmi:")
display(pd.DataFrame(previewRows))


## Eksplorasi Data 2D

Bagian ini melakukan pembacaan cepat terhadap struktur molekul dari `SMILES` menggunakan RDKit agar kita tetap punya gambaran ukuran graf yang masuk ke eksperimen sebelum feature engineering penuh dijalankan.


In [ ]:
def shorten_smiles(smiles, max_length=96):
    return smiles if len(smiles) <= max_length else smiles[: max_length - 3] + "..."


edaIndices = selectedTrainIndices[: min(edaMoleculeSampleCount, len(selectedTrainIndices))]
edaRows = []

for datasetIndex in edaIndices:
    smiles, targetValue = dataset[int(datasetIndex)]
    molecule = Chem.MolFromSmiles(smiles)
    if molecule is None:
        continue
    molecule = Chem.AddHs(molecule)
    edaRows.append(
        {
            "sampleIndex": int(datasetIndex),
            "judulMolekul": shorten_smiles(smiles),
            "atomCount": int(molecule.GetNumAtoms()),
            "bondCount": int(molecule.GetNumBonds()),
            targetColumnName: float(targetValue),
        }
    )

edaFrame = pd.DataFrame(edaRows)
if edaFrame.empty:
    raise RuntimeError("EDA gagal karena tidak ada molekul valid yang berhasil diparse dari SMILES.")

edaSummaryFrame = pd.DataFrame(
    {
        "metrik": [
            "jumlahSampelEDA",
            "atomCountRataRata",
            "bondCountRataRata",
            "atomCountMaksimum",
            "bondCountMaksimum",
        ],
        "nilai": [
            int(len(edaFrame)),
            float(edaFrame["atomCount"].mean()),
            float(edaFrame["bondCount"].mean()),
            float(edaFrame["atomCount"].max()),
            float(edaFrame["bondCount"].max()),
        ],
    }
)

print("Ringkasan eksplorasi data:")
display(edaSummaryFrame)
display(edaFrame.head(10))

fig, axes = plt.subplots(1, 3, figsize=(19, 5))
sns.histplot(edaFrame["atomCount"], bins=30, kde=True, ax=axes[0], color="#0f766e")
axes[0].set_title("Distribusi Jumlah Atom")
axes[0].set_xlabel("atomCount")

sns.histplot(edaFrame["bondCount"], bins=30, kde=True, ax=axes[1], color="#1d4ed8")
axes[1].set_title("Distribusi Jumlah Ikatan")
axes[1].set_xlabel("bondCount")

sns.scatterplot(
    data=edaFrame,
    x="atomCount",
    y="bondCount",
    alpha=0.35,
    ax=axes[2],
    color="#0f172a",
    edgecolor=None,
)
axes[2].set_title("Kepadatan Struktur Molekul")
axes[2].set_xlabel("atomCount")
axes[2].set_ylabel("bondCount")

plt.tight_layout()
plt.show()


## Feature Engineering dari SMILES

Bagian ini mengganti parser SDF lokal menjadi extractor fitur berbasis `SMILES` + RDKit. Nama file output dijaga tetap sama agar artefaknya masih bisa langsung dipakai oleh folder web lokal.


In [ ]:
elementNumberMap = {
    "H": 1,
    "C": 6,
    "N": 7,
    "O": 8,
    "F": 9,
    "P": 15,
    "S": 16,
    "Cl": 17,
    "Br": 35,
    "I": 53,
}

elementMassMap = {
    "H": 1.008,
    "C": 12.011,
    "N": 14.007,
    "O": 15.999,
    "F": 18.998,
    "P": 30.974,
    "S": 32.06,
    "Cl": 35.45,
    "Br": 79.904,
    "I": 126.904,
}


def build_rdkit_molecule(smiles, add_hydrogens=True):
    molecule = Chem.MolFromSmiles(smiles)
    if molecule is None:
        return None
    if add_hydrogens:
        molecule = Chem.AddHs(molecule)
    AllChem.Compute2DCoords(molecule)
    return molecule


def count_connected_components(atom_count, bond_pairs):
    if atom_count == 0:
        return 0

    adjacency_list = [[] for _ in range(atom_count)]
    for begin_index, end_index in bond_pairs:
        adjacency_list[begin_index].append(end_index)
        adjacency_list[end_index].append(begin_index)

    visited_flags = [False] * atom_count
    component_count = 0

    for start_index in range(atom_count):
        if visited_flags[start_index]:
            continue
        component_count += 1
        stack_list = [start_index]
        visited_flags[start_index] = True

        while stack_list:
            current_index = stack_list.pop()
            for neighbor_index in adjacency_list[current_index]:
                if not visited_flags[neighbor_index]:
                    visited_flags[neighbor_index] = True
                    stack_list.append(neighbor_index)

    return component_count


def compute_bond_length_stats(molecule, bond_pairs):
    if not bond_pairs:
        return 0.0, 0.0

    conformer = molecule.GetConformer()
    bond_lengths = []
    for begin_index, end_index in bond_pairs:
        begin_point = conformer.GetAtomPosition(begin_index)
        end_point = conformer.GetAtomPosition(end_index)
        bond_lengths.append(
            float(
                np.linalg.norm(
                    [
                        begin_point.x - end_point.x,
                        begin_point.y - end_point.y,
                        begin_point.z - end_point.z,
                    ]
                )
            )
        )

    return float(np.mean(bond_lengths)), float(np.std(bond_lengths))


def extract_molecule_features(smiles, dataset_index, target_value, split_name):
    molecule = build_rdkit_molecule(smiles, add_hydrogens=True)
    if molecule is None:
        return None

    atom_symbols = [atom.GetSymbol() for atom in molecule.GetAtoms()]
    atom_count = int(len(atom_symbols))
    if atom_count == 0:
        return None

    bond_pairs = []
    bond_orders = []
    degree_values = [0] * atom_count

    for bond in molecule.GetBonds():
        begin_index = bond.GetBeginAtomIdx()
        end_index = bond.GetEndAtomIdx()
        if bond.GetIsAromatic():
            bond_order = 4
        else:
            bond_order = int(round(float(bond.GetBondTypeAsDouble())))
            bond_order = max(1, min(bond_order, 3))

        bond_pairs.append((begin_index, end_index))
        bond_orders.append(bond_order)
        degree_values[begin_index] += 1
        degree_values[end_index] += 1

    element_counter = Counter(atom_symbols)
    atomic_numbers = np.array([elementNumberMap.get(symbol, 0) for symbol in atom_symbols], dtype=float)
    atomic_masses = np.array([elementMassMap.get(symbol, 0.0) for symbol in atom_symbols], dtype=float)
    degree_array = np.array(degree_values, dtype=float)

    heavy_atom_count = int(sum(symbol != "H" for symbol in atom_symbols))
    hydrogen_count = int(atom_count - heavy_atom_count)
    hetero_atom_count = int(sum(symbol not in {"C", "H"} for symbol in atom_symbols))
    component_count = count_connected_components(atom_count, bond_pairs)
    ring_index = max(len(bond_pairs) - atom_count + component_count, 0)
    mean_bond_length, std_bond_length = compute_bond_length_stats(molecule, bond_pairs)

    return {
        "moleculeIndex": int(dataset_index),
        "splitName": split_name,
        "moleculeTitle": shorten_smiles(smiles, max_length=140),
        "smiles": smiles,
        "atomCount": atom_count,
        "heavyAtomCount": heavy_atom_count,
        "hydrogenCount": hydrogen_count,
        "bondCount": len(bond_pairs),
        "singleBondCount": int(sum(order == 1 for order in bond_orders)),
        "doubleBondCount": int(sum(order == 2 for order in bond_orders)),
        "tripleBondCount": int(sum(order == 3 for order in bond_orders)),
        "aromaticBondCount": int(sum(order == 4 for order in bond_orders)),
        "uniqueElementCount": int(len(element_counter)),
        "heteroAtomCount": hetero_atom_count,
        "heteroAtomFraction": float(hetero_atom_count / atom_count),
        "totalAtomicNumber": float(atomic_numbers.sum()),
        "meanAtomicNumber": float(atomic_numbers.mean()),
        "totalAtomicMass": float(atomic_masses.sum()),
        "meanAtomicMass": float(atomic_masses.mean()),
        "degreeMean": float(degree_array.mean()),
        "degreeStd": float(degree_array.std()),
        "degreeMax": float(degree_array.max()),
        "degreeMin": float(degree_array.min()),
        "terminalAtomFraction": float(np.mean(degree_array == 1)),
        "branchAtomFraction": float(np.mean(degree_array >= 3)),
        "bondToAtomRatio": float(len(bond_pairs) / atom_count),
        "ringIndex": float(ring_index),
        "meanBondLength": float(mean_bond_length),
        "stdBondLength": float(std_bond_length),
        "elementCountC": int(element_counter.get("C", 0)),
        "elementCountH": int(element_counter.get("H", 0)),
        "elementCountN": int(element_counter.get("N", 0)),
        "elementCountO": int(element_counter.get("O", 0)),
        "elementCountF": int(element_counter.get("F", 0)),
        "elementCountP": int(element_counter.get("P", 0)),
        "elementCountS": int(element_counter.get("S", 0)),
        "elementCountCl": int(element_counter.get("Cl", 0)),
        "elementCountBr": int(element_counter.get("Br", 0)),
        "elementCountI": int(element_counter.get("I", 0)),
        targetColumnName: float(target_value) if pd.notna(target_value) else np.nan,
    }


featureCsvPath = outputDir / "fiturRegresiSdf.csv"
featureMetadataPath = outputDir / "fiturRegresiSdf.metadata.json"

cacheIsReusable = False
cachedFeatureMetadata = {}

if reuseCachedFeatures and featureCsvPath.exists() and featureMetadataPath.exists():
    with featureMetadataPath.open("r", encoding="utf-8") as handle:
        cachedFeatureMetadata = json.load(handle)

    cacheIsReusable = (
        cachedFeatureMetadata.get("sdfPath") == "ogb://pcqm4mv2?only_smiles=true"
        and int(cachedFeatureMetadata.get("trainSampleCount", -1)) == int(len(selectedTrainIndices))
        and int(cachedFeatureMetadata.get("validSampleCount", -1)) == int(len(selectedValidIndices))
        and cachedFeatureMetadata.get("targetColumnName") == targetColumnName
    )

if cacheIsReusable:
    featureFrame = pd.read_csv(featureCsvPath)
    skippedMoleculeCount = int(cachedFeatureMetadata.get("skippedMoleculeCount", 0))
    elapsedSeconds = float(cachedFeatureMetadata.get("elapsedSeconds", 0.0))
    print("Memuat fitur dari cache yang cocok dengan konfigurasi Kaggle saat ini.")
else:
    startTime = time.perf_counter()
    featureRows = []
    skippedMoleculeCount = 0

    experimentSplits = [
        ("train", selectedTrainIndices),
        ("valid", selectedValidIndices),
    ]

    for splitName, splitIndices in experimentSplits:
        print(f"Mulai ekstraksi fitur untuk split {splitName} ({len(splitIndices):,} molekul).")
        for rowNumber, datasetIndex in enumerate(splitIndices, start=1):
            smiles, targetValue = dataset[int(datasetIndex)]
            featureRow = extract_molecule_features(smiles, int(datasetIndex), targetValue, splitName)
            if featureRow is None:
                skippedMoleculeCount += 1
                continue
            featureRows.append(featureRow)

            if reportInterval and rowNumber % reportInterval == 0:
                print(
                    f"[{splitName}] sudah memproses {rowNumber:,} molekul "
                    f"pada indeks dataset terakhir {int(datasetIndex):,}."
                )

    featureFrame = pd.DataFrame(featureRows)
    elapsedSeconds = time.perf_counter() - startTime

    if featureFrame.empty:
        raise RuntimeError("Tidak ada fitur yang berhasil diekstrak dari split resmi OGB.")

    featureFrame = (
        featureFrame
        .sort_values(["splitName", "moleculeIndex"])
        .reset_index(drop=True)
    )

    featureFrame.to_csv(featureCsvPath, index=False)
    cachedFeatureMetadata = {
        "sdfPath": "ogb://pcqm4mv2?only_smiles=true",
        "datasetRoot": str(datasetRoot),
        "trainSampleCount": int(len(selectedTrainIndices)),
        "validSampleCount": int(len(selectedValidIndices)),
        "sampleStride": int(sampleStride),
        "targetColumnName": targetColumnName,
        "splitProtocol": "official-ogb-train-valid",
        "skippedMoleculeCount": int(skippedMoleculeCount),
        "elapsedSeconds": float(elapsedSeconds),
    }
    with featureMetadataPath.open("w", encoding="utf-8") as handle:
        json.dump(cachedFeatureMetadata, handle, ensure_ascii=False, indent=2)

modelFeatureColumns = DEFAULT_FEATURE_COLUMNS.copy()

preprocessSummaryFrame = pd.DataFrame(
    {
        "informasi": [
            "jumlahBarisFitur",
            "jumlahKolomFitur",
            "nilaiHilangTotal",
            "skippedMoleculeCount",
            "waktuPreprocessDetik",
            "targetMinimum",
            "targetMaksimum",
            "targetRataRata",
        ],
        "nilai": [
            int(len(featureFrame)),
            int(featureFrame.shape[1]),
            int(featureFrame.isna().sum().sum()),
            int(skippedMoleculeCount),
            float(elapsedSeconds),
            float(featureFrame[targetColumnName].min()),
            float(featureFrame[targetColumnName].max()),
            float(featureFrame[targetColumnName].mean()),
        ],
    }
)

print("Ringkasan hasil preprocessing:")
display(preprocessSummaryFrame)
display(featureFrame.head())
print(f"Fitur tersimpan di: {featureCsvPath}")


## Split Resmi OGB

Bagian ini menyiapkan matriks fitur train dan validasi berdasarkan split resmi OGB. File prediksi evaluasi tetap dinamai seperti versi lokal agar output-nya bisa langsung menimpa folder lama.


In [ ]:
trainFrame = featureFrame[featureFrame["splitName"] == "train"].reset_index(drop=True).copy()
validFrame = featureFrame[featureFrame["splitName"] == "valid"].reset_index(drop=True).copy()

xTrain = trainFrame[modelFeatureColumns]
yTrain = trainFrame[targetColumnName]
xValid = validFrame[modelFeatureColumns]
yValid = validFrame[targetColumnName]

splitFrame = pd.DataFrame(
    {
        "bagianData": ["train", "validasi", "test-dev", "test-challenge"],
        "jumlahBaris": [
            int(len(selectedTrainIndices)),
            int(len(selectedValidIndices)),
            int(len(splitDict["test-dev"])),
            int(len(splitDict["test-challenge"])),
        ],
        "fraksi": [
            float(len(selectedTrainIndices) / totalDatasetCount),
            float(len(selectedValidIndices) / totalDatasetCount),
            float(len(splitDict["test-dev"]) / totalDatasetCount),
            float(len(splitDict["test-challenge"]) / totalDatasetCount),
        ],
    }
)

print("Ringkasan split resmi OGB yang dipakai:")
display(splitFrame)


## Training Baseline

Bagian ini membandingkan model regresi tabular ringan di atas fitur hasil RDKit. Fokusnya adalah baseline yang cukup realistis untuk dijalankan di Kaggle tanpa bergantung pada file SDF lokal.


In [ ]:
modelMap = build_model_map(randomState)

resultRows = []
trainedModelMap = {}
graphModelResult = None

for modelName, modelObject in modelMap.items():
    startTime = time.perf_counter()
    try:
        trainedModel = clone(modelObject)
        trainedModel.fit(xTrain, yTrain)
        validPrediction = trainedModel.predict(xValid)
        durationSeconds = time.perf_counter() - startTime

        trainedModelMap[modelName] = trainedModel
        resultRows.append(
            {
                "modelName": modelName,
                "validMae": float(mean_absolute_error(yValid, validPrediction)),
                "validRmse": float(math.sqrt(mean_squared_error(yValid, validPrediction))),
                "validR2": float(r2_score(yValid, validPrediction)),
                "durasiDetik": float(durationSeconds),
            }
        )
        print(f"Model {modelName} selesai dilatih.")
    except Exception as modelError:
        print(f"Model {modelName} dilewati: {modelError}")

if not resultRows:
    raise RuntimeError("Tidak ada model yang berhasil dilatih pada subset Kaggle ini.")

resultFrame = pd.DataFrame(resultRows).sort_values("validMae").reset_index(drop=True)
bestModelName = str(resultFrame.loc[0, "modelName"])

print("Perbandingan model pada split validasi resmi:")
display(resultFrame)
print(f"Model terbaik sementara: {bestModelName}")


## Evaluasi pada Validasi Resmi

Karena label `test-dev` dan `test-challenge` tidak tersedia, evaluasi metrik di notebook ini dilakukan pada split `valid` resmi OGB. Jika kamu butuh submission `.npz`, flag di sel penyimpanan bisa diaktifkan untuk `test-dev`.


In [ ]:
graphHistoryFrame = pd.DataFrame(columns=["epoch", "trainLoss", "validMae", "validRmse"])

bestModel = trainedModelMap[bestModelName]
validPrediction = bestModel.predict(xValid)

predictionFrame = validFrame[
    ["moleculeIndex", "moleculeTitle", "smiles", "atomCount", "bondCount"]
].copy()
predictionFrame["splitName"] = "valid"
predictionFrame["nilaiAktual"] = yValid.to_numpy()
predictionFrame["nilaiPrediksi"] = validPrediction
predictionFrame["galatAbsolut"] = (
    predictionFrame["nilaiAktual"] - predictionFrame["nilaiPrediksi"]
).abs()
predictionFrame["residual"] = (
    predictionFrame["nilaiPrediksi"] - predictionFrame["nilaiAktual"]
)
predictionFrame["galatKuadrat"] = predictionFrame["residual"] ** 2
predictionFrame["galatRelatifAbsolut"] = (
    predictionFrame["galatAbsolut"] / np.maximum(predictionFrame["nilaiAktual"].abs(), 1e-8)
)

testMae = float(mean_absolute_error(predictionFrame["nilaiAktual"], predictionFrame["nilaiPrediksi"]))
testMse = float(mean_squared_error(predictionFrame["nilaiAktual"], predictionFrame["nilaiPrediksi"]))
testRmse = float(math.sqrt(testMse))
testR2 = float(r2_score(predictionFrame["nilaiAktual"], predictionFrame["nilaiPrediksi"]))
testMedAe = float(
    median_absolute_error(predictionFrame["nilaiAktual"], predictionFrame["nilaiPrediksi"])
)
testExplainedVariance = float(
    explained_variance_score(predictionFrame["nilaiAktual"], predictionFrame["nilaiPrediksi"])
)
testMape = float(predictionFrame["galatRelatifAbsolut"].mean())
testSmape = float(
    np.mean(
        2.0
        * predictionFrame["galatAbsolut"]
        / np.maximum(
            predictionFrame["nilaiAktual"].abs() + predictionFrame["nilaiPrediksi"].abs(),
            1e-8,
        )
    )
)
pearsonCorrelation = float(
    np.corrcoef(predictionFrame["nilaiAktual"], predictionFrame["nilaiPrediksi"])[0, 1]
) if len(predictionFrame) > 1 else np.nan

officialValidMae = float(
    evaluator.eval(
        {
            "y_true": predictionFrame["nilaiAktual"].to_numpy(dtype=np.float32),
            "y_pred": predictionFrame["nilaiPrediksi"].to_numpy(dtype=np.float32),
        }
    )["mae"]
)

metricFrame = pd.DataFrame(
    {
        "metrik": [
            "MAE",
            "OGBValidMAE",
            "MSE",
            "RMSE",
            "MedianAE",
            "MAPE",
            "sMAPE",
            "R2",
            "ExplainedVariance",
            "PearsonCorrelation",
            "jumlahDataEvaluasi",
        ],
        "nilai": [
            testMae,
            officialValidMae,
            testMse,
            testRmse,
            testMedAe,
            testMape,
            testSmape,
            testR2,
            testExplainedVariance,
            pearsonCorrelation,
            float(len(predictionFrame)),
        ],
    }
)

residualSummaryFrame = pd.DataFrame(
    {
        "ringkasan": ["meanResidual", "stdResidual", "minResidual", "maxResidual"],
        "nilai": [
            float(predictionFrame["residual"].mean()),
            float(predictionFrame["residual"].std()),
            float(predictionFrame["residual"].min()),
            float(predictionFrame["residual"].max()),
        ],
    }
)

evaluationDataSource = "fiturSmilesOfficial"

print("Hasil evaluasi pada split validasi resmi:")
display(metricFrame)
print("Contoh prediksi pada split validasi:")
display(predictionFrame.head(10))
print("Ringkasan residual:")
display(residualSummaryFrame)


## Visualisasi Hasil

Bagian ini menyimpan plot yang tetap dipakai website lokal, tetapi judul dan targetnya sudah mengikuti tugas resmi OGB berbasis `HOMO-LUMO gap`.


In [ ]:
plotPathMap = {}

correlationSeries = (
    featureFrame[modelFeatureColumns + [targetColumnName]]
    .corr(numeric_only=True)[targetColumnName]
    .drop(targetColumnName)
    .abs()
    .sort_values(ascending=False)
    .head(12)
)
correlationFrame = correlationSeries.rename("korelasiAbsolut").reset_index()
correlationFrame.columns = ["featureName", "korelasiAbsolut"]

modelComparisonFigurePath = figureDir / "perbandingan-model-validasi.png"
evaluationFigurePath = figureDir / "evaluasi-model-terbaik.png"
regressionDiagnosticFigurePath = figureDir / "diagnostik-regresi.png"
featureSignalFigurePath = figureDir / "sinyal-fitur.png"
graphHistoryFigurePath = figureDir / "riwayat-training-gin.png"

plt.figure(figsize=(10, 5.5))
sns.barplot(data=resultFrame.sort_values("validMae"), x="validMae", y="modelName", color="#0f766e")
plt.title("Perbandingan Model pada Validasi Resmi OGB")
plt.xlabel("MAE Validasi")
plt.ylabel("Model")
plt.tight_layout()
plt.savefig(modelComparisonFigurePath, dpi=180, bbox_inches="tight")
plotPathMap["modelComparison"] = str(modelComparisonFigurePath)
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(featureFrame[targetColumnName], bins=40, kde=True, ax=axes[0, 0], color="#0f766e")
axes[0, 0].set_title("Distribusi Target HOMO-LUMO Gap")
axes[0, 0].set_xlabel(targetColumnName)
axes[0, 0].set_ylabel("Frekuensi")

axes[0, 1].scatter(
    predictionFrame["nilaiAktual"],
    predictionFrame["nilaiPrediksi"],
    alpha=0.35,
    color="#0f172a",
)
minimumValue = min(
    predictionFrame["nilaiAktual"].min(),
    predictionFrame["nilaiPrediksi"].min(),
)
maximumValue = max(
    predictionFrame["nilaiAktual"].max(),
    predictionFrame["nilaiPrediksi"].max(),
)
axes[0, 1].plot([minimumValue, maximumValue], [minimumValue, maximumValue], linestyle="--", color="black")
axes[0, 1].set_title("Aktual vs Prediksi pada Validasi")
axes[0, 1].set_xlabel("Nilai Aktual")
axes[0, 1].set_ylabel("Nilai Prediksi")

sns.histplot(predictionFrame["galatAbsolut"], bins=35, kde=True, ax=axes[1, 0], color="#1d4ed8")
axes[1, 0].set_title("Distribusi Galat Absolut")
axes[1, 0].set_xlabel("galatAbsolut")
axes[1, 0].set_ylabel("Frekuensi")

axes[1, 1].scatter(
    predictionFrame["nilaiPrediksi"],
    predictionFrame["residual"],
    alpha=0.35,
    color="#7c3aed",
)
axes[1, 1].axhline(0.0, linestyle="--", color="black")
axes[1, 1].set_title("Residual vs Prediksi")
axes[1, 1].set_xlabel("Nilai Prediksi")
axes[1, 1].set_ylabel("Residual")

plt.tight_layout()
plt.savefig(evaluationFigurePath, dpi=180, bbox_inches="tight")
plotPathMap["evaluation"] = str(evaluationFigurePath)
plt.show()

sortedPredictionFrame = predictionFrame.sort_values("nilaiAktual").reset_index(drop=True)
quantileGrid = np.linspace(0.05, 0.95, 19)
quantileFrame = pd.DataFrame(
    {
        "kuantil": quantileGrid,
        "aktual": np.quantile(predictionFrame["nilaiAktual"], quantileGrid),
        "prediksi": np.quantile(predictionFrame["nilaiPrediksi"], quantileGrid),
    }
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(
    sortedPredictionFrame.index,
    sortedPredictionFrame["nilaiAktual"],
    label="Aktual",
    color="#0f172a",
)
axes[0].plot(
    sortedPredictionFrame.index,
    sortedPredictionFrame["nilaiPrediksi"],
    label="Prediksi",
    color="#0f766e",
    alpha=0.85,
)
axes[0].set_title("Perbandingan Aktual dan Prediksi Terurut")
axes[0].set_xlabel("Urutan Sampel Validasi")
axes[0].set_ylabel("Nilai Target")
axes[0].legend()

axes[1].plot(quantileFrame["kuantil"], quantileFrame["aktual"], marker="o", label="Aktual", color="#0f172a")
axes[1].plot(quantileFrame["kuantil"], quantileFrame["prediksi"], marker="o", label="Prediksi", color="#1d4ed8")
axes[1].set_title("Perbandingan Kuantil Aktual vs Prediksi")
axes[1].set_xlabel("Kuantil")
axes[1].set_ylabel("Nilai Target")
axes[1].legend()

plt.tight_layout()
plt.savefig(regressionDiagnosticFigurePath, dpi=180, bbox_inches="tight")
plotPathMap["regressionDiagnostic"] = str(regressionDiagnosticFigurePath)
plt.show()

print("Fitur dengan korelasi absolut tertinggi terhadap target:")
display(correlationFrame)

importanceFrame = build_importance_frame(bestModel, modelFeatureColumns)
importanceExportFrame = importanceFrame.copy()

if importanceFrame.empty:
    featureSignalFrame = correlationFrame.rename(columns={"korelasiAbsolut": "score"}).copy()
    featureSignalTitle = "12 Fitur dengan Korelasi Tertinggi"
    featureSignalLabel = "Korelasi Absolut"
    print("Model terbaik tidak menyediakan feature importance, sehingga plot fitur memakai korelasi absolut.")
else:
    featureSignalFrame = importanceFrame.rename(columns={"importance": "score"}).copy()
    featureSignalTitle = "12 Fitur Terpenting pada Model Terbaik"
    featureSignalLabel = "Besaran Importance"
    display(importanceFrame.head(15))

plt.figure(figsize=(9, 6))
sns.barplot(data=featureSignalFrame.head(12), x="score", y="featureName", color="#0f766e")
plt.title(featureSignalTitle)
plt.xlabel(featureSignalLabel)
plt.ylabel("Nama Fitur")
plt.tight_layout()
plt.savefig(featureSignalFigurePath, dpi=180, bbox_inches="tight")
plotPathMap["featureSignal"] = str(featureSignalFigurePath)
plt.show()

if not graphHistoryFrame.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    axes[0].plot(graphHistoryFrame["epoch"], graphHistoryFrame["trainLoss"], marker="o", color="#0f766e")
    axes[0].set_title("Train Loss GIN per Epoch")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Smooth L1 Loss")

    axes[1].plot(graphHistoryFrame["epoch"], graphHistoryFrame["validMae"], marker="o", color="#1d4ed8", label="Valid MAE")
    axes[1].plot(graphHistoryFrame["epoch"], graphHistoryFrame["validRmse"], marker="o", color="#dc2626", label="Valid RMSE")
    axes[1].set_title("Validasi GIN per Epoch")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Nilai Metrik")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(graphHistoryFigurePath, dpi=180, bbox_inches="tight")
    plotPathMap["ginTrainingHistory"] = str(graphHistoryFigurePath)
    plt.show()


## Penyimpanan Hasil dan Paket Download

Bagian ini menulis artefak akhir dengan nama file yang sama seperti versi lokal, menyiapkan JSON untuk website, membuat preview viewer ringan, dan men-zip folder `output` supaya gampang diunduh dari Kaggle.


In [ ]:
artifactPath = outputDir / "modelRegresiRadiusGyration.pkl"
predictionPath = outputDir / "prediksiDataTest.csv"
metadataPath = outputDir / "ringkasanEksperimen.json"
modelComparisonPath = outputDir / "perbandinganModelValidasi.csv"
metricPath = outputDir / "metrikEvaluasiModelTerbaik.csv"
correlationPath = outputDir / "korelasiFitur.csv"
importancePath = outputDir / "fiturTerpenting.csv"
graphHistoryPath = outputDir / "riwayatTrainingGin.csv"
residualSummaryPath = outputDir / "ringkasanResidual.csv"
quantileDiagnosticPath = outputDir / "diagnostikKuantilRegresi.csv"
plotManifestPath = outputDir / "daftarPlotEvaluasi.json"
presentationSummaryPath = outputDir / "ringkasanPresentasi.json"
showcasePath = outputDir / "notebookShowcase.json"
webManifestPath = outputDir / "notebookWebManifest.json"
overview3dPath = outputDir / "overviewMolekul3d.csv"
downloadZipPath = Path("output_kaggle_ready.zip")


def round_coord(value, digits):
    rounded = round(float(value), digits)
    return 0.0 if rounded == -0.0 else rounded


def build_palette(symbols):
    return {
        symbol: {
            "color": DEFAULT_ELEMENT_COLORS.get(symbol, "#475569"),
            "size": DEFAULT_ELEMENT_SIZES.get(symbol, 1.55),
        }
        for symbol in symbols
    }


def export_smiles_viewer(preview_frame, output_dir, molecules_per_row=20, cell_gap=8.0, round_digits=3):
    output_dir.mkdir(parents=True, exist_ok=True)

    for existing_chunk in output_dir.glob("chunk-*.json"):
        existing_chunk.unlink()

    chunk_nodes = []
    chunk_edges = []
    chunk_molecules = []
    chunk_records = []
    symbols = []
    symbol_to_index = {}
    element_counts = {}

    total_nodes = 0
    total_edges = 0
    bounds_min_x = None
    bounds_max_x = None
    bounds_min_y = None
    bounds_max_y = None
    bounds_min_z = None
    bounds_max_z = None

    cursor_x = 0.0
    cursor_z = 0.0
    row_depth = 0.0

    for exportIndex, previewRow in enumerate(preview_frame.to_dict(orient="records")):
        molecule = build_rdkit_molecule(previewRow["smiles"], add_hydrogens=False)
        if molecule is None or molecule.GetNumAtoms() == 0:
            continue

        if exportIndex and exportIndex % molecules_per_row == 0:
            cursor_x = 0.0
            cursor_z -= row_depth + cell_gap
            row_depth = 0.0

        conformer = molecule.GetConformer()
        x_values = [conformer.GetAtomPosition(atom.GetIdx()).x for atom in molecule.GetAtoms()]
        y_values = [conformer.GetAtomPosition(atom.GetIdx()).y for atom in molecule.GetAtoms()]
        min_x = min(x_values)
        max_x = max(x_values)
        min_y = min(y_values)
        max_y = max(y_values)
        center_x = (min_x + max_x) / 2.0
        center_y = (min_y + max_y) / 2.0
        molecule_width = max(max_x - min_x, 1.0)
        molecule_depth = max(max_y - min_y, 1.0)

        node_ids = []
        for atom in molecule.GetAtoms():
            atom_index = atom.GetIdx()
            point = conformer.GetAtomPosition(atom_index)
            symbol = atom.GetSymbol()
            if symbol not in symbol_to_index:
                symbol_to_index[symbol] = len(symbols)
                symbols.append(symbol)

            plot_x = round_coord((point.x - center_x) + cursor_x, round_digits)
            plot_y = 0.0
            plot_z = round_coord((point.y - center_y) + cursor_z, round_digits)
            node_id = f"m{exportIndex}a{atom_index}"

            chunk_nodes.append(
                [node_id, plot_x, plot_y, plot_z, symbol_to_index[symbol], exportIndex]
            )
            node_ids.append(node_id)
            total_nodes += 1
            element_counts[symbol] = element_counts.get(symbol, 0) + 1

            bounds_min_x = plot_x if bounds_min_x is None else min(bounds_min_x, plot_x)
            bounds_max_x = plot_x if bounds_max_x is None else max(bounds_max_x, plot_x)
            bounds_min_y = plot_y if bounds_min_y is None else min(bounds_min_y, plot_y)
            bounds_max_y = plot_y if bounds_max_y is None else max(bounds_max_y, plot_y)
            bounds_min_z = plot_z if bounds_min_z is None else min(bounds_min_z, plot_z)
            bounds_max_z = plot_z if bounds_max_z is None else max(bounds_max_z, plot_z)

        for bond_index, bond in enumerate(molecule.GetBonds()):
            begin_index = bond.GetBeginAtomIdx()
            end_index = bond.GetEndAtomIdx()
            chunk_edges.append(
                [f"m{exportIndex}e{bond_index}", node_ids[begin_index], node_ids[end_index], exportIndex]
            )
            total_edges += 1

        chunk_molecules.append(
            [
                exportIndex,
                previewRow["moleculeTitle"],
                int(molecule.GetNumAtoms()),
                int(molecule.GetNumBonds()),
            ]
        )

        cursor_x += molecule_width + cell_gap
        row_depth = max(row_depth, molecule_depth)

    chunk_path = output_dir / "chunk-00000.json"
    chunk_payload = {
        "chunkIndex": 0,
        "molecules": chunk_molecules,
        "nodes": chunk_nodes,
        "edges": chunk_edges,
    }
    chunk_path.write_text(json.dumps(chunk_payload, ensure_ascii=True, separators=(",", ":")), encoding="utf-8")

    if chunk_molecules:
        chunk_records.append(
            {
                "file": chunk_path.name,
                "molecules": len(chunk_molecules),
                "nodes": len(chunk_nodes),
                "edges": len(chunk_edges),
                "firstMoleculeIndex": int(chunk_molecules[0][0]),
                "lastMoleculeIndex": int(chunk_molecules[-1][0]),
            }
        )

    manifest_payload = {
        "formatVersion": 2,
        "generatedAt": datetime.now(timezone.utc).isoformat(),
        "dataset": {
            "sourceFile": "pcqm4mv2-ogb-network",
            "sourceBytes": 0,
        },
        "export": {
            "chunkSize": len(chunk_molecules),
            "stride": 1,
            "maxMolecules": len(chunk_molecules),
            "indexMode": "sequential",
            "layoutMode": "layered-2d-from-smiles",
            "depthScale": 1.0,
            "roundDigits": round_digits,
            "includeHydrogens": False,
            "randomSeed": randomState,
            "clusterCount": 1,
            "clusterRadius": 0.0,
            "clusterSpread": 0.0,
            "moleculesPerRow": molecules_per_row,
            "rowsPerLayer": 1,
            "cellGap": cell_gap,
            "layerGap": 0.0,
        },
        "summary": {
            "seenMolecules": len(chunk_molecules),
            "exportedMolecules": len(chunk_molecules),
            "skippedMolecules": int(len(preview_frame) - len(chunk_molecules)),
            "nodes": total_nodes,
            "edges": total_edges,
            "averageAtomsPerMolecule": round(total_nodes / max(len(chunk_molecules), 1), 3),
            "averageEdgesPerMolecule": round(total_edges / max(len(chunk_molecules), 1), 3),
            "minAtoms": min([row[2] for row in chunk_molecules], default=0),
            "maxAtoms": max([row[2] for row in chunk_molecules], default=0),
            "minEdges": min([row[3] for row in chunk_molecules], default=0),
            "maxEdges": max([row[3] for row in chunk_molecules], default=0),
            "atomsBeforeFilter": total_nodes,
            "bondsBeforeFilter": total_edges,
        },
        "layoutBounds": {
            "minX": bounds_min_x if bounds_min_x is not None else 0.0,
            "maxX": bounds_max_x if bounds_max_x is not None else 0.0,
            "minY": bounds_min_y if bounds_min_y is not None else 0.0,
            "maxY": bounds_max_y if bounds_max_y is not None else 0.0,
            "minZ": bounds_min_z if bounds_min_z is not None else 0.0,
            "maxZ": bounds_max_z if bounds_max_z is not None else 0.0,
        },
        "symbols": symbols,
        "elementCounts": element_counts,
        "elementStyle": build_palette(symbols),
        "chunks": chunk_records,
    }
    manifest_path = output_dir / "manifest.json"
    manifest_path.write_text(json.dumps(manifest_payload, ensure_ascii=False, indent=2), encoding="utf-8")
    return manifest_path


metricExportFrame = metricFrame.copy()
metricExportFrame["bestModelName"] = bestModelName
metricExportFrame["evaluationDataSource"] = evaluationDataSource

artifactPayload = {
    "bestModelName": bestModelName,
    "targetColumnName": targetColumnName,
    "featureColumnNames": modelFeatureColumns,
    "modelObject": bestModel,
    "testMetrics": {
        "mae": float(testMae),
        "officialValidMae": float(officialValidMae),
        "mse": float(testMse),
        "rmse": float(testRmse),
        "medianAe": float(testMedAe),
        "mape": float(testMape),
        "smape": float(testSmape),
        "r2": float(testR2),
        "explainedVariance": float(testExplainedVariance),
        "pearsonCorrelation": float(pearsonCorrelation),
    },
    "evaluationDataSource": evaluationDataSource,
    "workflowType": "pcqm4mv2-ogb-smiles-baseline",
    "plotPaths": plotPathMap,
}

with artifactPath.open("wb") as handle:
    pickle.dump(artifactPayload, handle)

predictionFrame.to_csv(predictionPath, index=False)
resultFrame.to_csv(modelComparisonPath, index=False)
metricExportFrame.to_csv(metricPath, index=False)
correlationFrame.to_csv(correlationPath, index=False)
importanceExportFrame.to_csv(importancePath, index=False)
graphHistoryFrame.to_csv(graphHistoryPath, index=False)
residualSummaryFrame.to_csv(residualSummaryPath, index=False)
quantileFrame.to_csv(quantileDiagnosticPath, index=False)

with plotManifestPath.open("w", encoding="utf-8") as handle:
    json.dump(plotPathMap, handle, ensure_ascii=False, indent=2)

overview3dFrame = featureFrame[
    ["moleculeIndex", "atomCount", "heavyAtomCount", "bondCount", targetColumnName]
].copy()
overview3dFrame["posX"] = (
    overview3dFrame["atomCount"].astype(float) - float(overview3dFrame["atomCount"].mean())
)
overview3dFrame["posY"] = (
    overview3dFrame["bondCount"].astype(float) - float(overview3dFrame["bondCount"].mean())
)
overview3dFrame["posZ"] = (
    overview3dFrame[targetColumnName].astype(float) - float(overview3dFrame[targetColumnName].mean())
) * 18.0
overview3dFrame.to_csv(overview3dPath, index=False)

submissionArtifactPath = None
if createOfficialTestDevSubmission:
    submissionModel = clone(modelMap[bestModelName])
    submissionTrainFrame = pd.concat([trainFrame, validFrame], axis=0, ignore_index=True)
    submissionModel.fit(
        submissionTrainFrame[modelFeatureColumns],
        submissionTrainFrame[targetColumnName],
    )

    testDevRows = []
    print("Mulai membangun fitur untuk test-dev resmi OGB. Ini bisa lama di Kaggle.")
    for rowNumber, datasetIndex in enumerate(np.asarray(splitDict["test-dev"], dtype=np.int64), start=1):
        smiles, _ = dataset[int(datasetIndex)]
        featureRow = extract_molecule_features(smiles, int(datasetIndex), np.nan, "test-dev")
        if featureRow is None:
            continue
        testDevRows.append(featureRow)
        if reportInterval and rowNumber % reportInterval == 0:
            print(f"[test-dev] sudah memproses {rowNumber:,} molekul.")

    testDevFeatureFrame = pd.DataFrame(testDevRows)
    testDevPredictions = submissionModel.predict(testDevFeatureFrame[modelFeatureColumns])
    submissionDir = outputDir / "submission_test-dev"
    submissionDir.mkdir(parents=True, exist_ok=True)
    evaluator.save_test_submission(
        input_dict={"y_pred": np.asarray(testDevPredictions, dtype=np.float32)},
        dir_path=str(submissionDir),
        mode="test-dev",
    )
    submissionArtifactPath = submissionDir / "y_pred_pcqm4m-v2_test-dev.npz"

presentationSummaryPayload = {
    "judul": "Ringkasan Evaluasi Baseline PCQM4Mv2",
    "bestModelName": bestModelName,
    "evaluationDataSource": evaluationDataSource,
    "targetColumnName": targetColumnName,
    "evaluationSplitName": "valid",
    "jumlahData": int(len(featureFrame)),
    "jumlahDataTest": int(len(predictionFrame)),
    "topModelsByValidMae": json.loads(resultFrame.head(5).round(6).to_json(orient="records")),
    "testMetrics": {
        "mae": float(testMae),
        "officialValidMae": float(officialValidMae),
        "mse": float(testMse),
        "rmse": float(testRmse),
        "medianAe": float(testMedAe),
        "mape": float(testMape),
        "smape": float(testSmape),
        "r2": float(testR2),
        "explainedVariance": float(testExplainedVariance),
        "pearsonCorrelation": float(pearsonCorrelation),
    },
    "featureSignalPreview": json.loads(featureSignalFrame.head(12).round(6).to_json(orient="records")),
    "savedTables": {
        "modelComparisonPath": str(modelComparisonPath),
        "metricPath": str(metricPath),
        "predictionPath": str(predictionPath),
        "correlationPath": str(correlationPath),
        "importancePath": str(importancePath),
        "graphHistoryPath": str(graphHistoryPath),
        "residualSummaryPath": str(residualSummaryPath),
        "quantileDiagnosticPath": str(quantileDiagnosticPath),
    },
    "savedPlots": plotPathMap,
}
with presentationSummaryPath.open("w", encoding="utf-8") as handle:
    json.dump(presentationSummaryPayload, handle, ensure_ascii=False, indent=2)

metadataPayload = {
    "sdfPath": "ogb://pcqm4mv2?only_smiles=true",
    "jumlahData": int(len(featureFrame)),
    "bestModelName": bestModelName,
    "targetColumnName": targetColumnName,
    "evaluationDataSource": evaluationDataSource,
    "evaluationSplitName": "valid",
    "evaluationProtocol": "official train/valid split dari OGB PCQM4Mv2, dieksekusi sebagai baseline Kaggle berbasis SMILES",
    "officialReferences": [
        "https://ogb.stanford.edu/docs/lsc/pcqm4mv2/",
        "https://ogb.stanford.edu/docs/lsc/leaderboards/#pcqm4mv2",
    ],
    "testMetrics": {
        "mae": float(testMae),
        "officialValidMae": float(officialValidMae),
        "mse": float(testMse),
        "rmse": float(testRmse),
        "medianAe": float(testMedAe),
        "mape": float(testMape),
        "smape": float(testSmape),
        "r2": float(testR2),
        "explainedVariance": float(testExplainedVariance),
        "pearsonCorrelation": float(pearsonCorrelation),
    },
    "savedTables": {
        "modelComparisonPath": str(modelComparisonPath),
        "metricPath": str(metricPath),
        "predictionPath": str(predictionPath),
        "correlationPath": str(correlationPath),
        "importancePath": str(importancePath),
        "graphHistoryPath": str(graphHistoryPath),
        "residualSummaryPath": str(residualSummaryPath),
        "quantileDiagnosticPath": str(quantileDiagnosticPath),
        "presentationSummaryPath": str(presentationSummaryPath),
    },
    "savedPlots": plotPathMap,
    "submissionArtifactPath": str(submissionArtifactPath) if submissionArtifactPath else None,
    "trainSampleCount": int(len(selectedTrainIndices)),
    "validSampleCount": int(len(selectedValidIndices)),
}
with metadataPath.open("w", encoding="utf-8") as handle:
    json.dump(metadataPayload, handle, ensure_ascii=False, indent=2)

viewerPreviewFrame = featureFrame.head(min(viewerMoleculeCount, len(featureFrame)))[
    ["moleculeIndex", "moleculeTitle", "smiles"]
].copy()
viewerManifestPath = export_smiles_viewer(viewerPreviewFrame, viewerDataDir)

statisticsColumns = ["mean", "std", "min", "25%", "50%", "75%", "max"]
statisticsFullFrame = (
    featureFrame[modelFeatureColumns + [targetColumnName]]
    .describe()
    .T[statisticsColumns]
    .reset_index()
    .rename(columns={"index": "featureName"})
)

testReferenceFrame = validFrame[
    ["moleculeIndex", "moleculeTitle", "atomCount", "heavyAtomCount", "hydrogenCount", "bondCount", targetColumnName]
].rename(columns={targetColumnName: "nilaiAktual"}).reset_index(drop=True)
predictionOverlayFrame = predictionFrame[["nilaiPrediksi", "galatAbsolut"]].reset_index(drop=True)
testMoleculePreview = pd.concat([testReferenceFrame, predictionOverlayFrame], axis=1).head(8)

fullTablePaths = {
    "overview3dPath": "output/overviewMolekul3d.csv",
    "featurePath": "output/fiturRegresiSdf.csv",
    "predictionPath": "output/prediksiDataTest.csv",
    "modelComparisonPath": "output/perbandinganModelValidasi.csv",
    "metricPath": "output/metrikEvaluasiModelTerbaik.csv",
    "correlationPath": "output/korelasiFitur.csv",
    "importancePath": "output/fiturTerpenting.csv",
    "graphHistoryPath": "output/riwayatTrainingGin.csv",
    "residualSummaryPath": "output/ringkasanResidual.csv",
    "quantileDiagnosticPath": "output/diagnostikKuantilRegresi.csv",
}

tableCatalog = [
    {
        "tableKey": "overviewMolekul3d",
        "title": "Overview 3D Molekul",
        "description": "Cloud 3D ringan untuk seluruh molekul eksperimen berbasis SMILES.",
        "path": fullTablePaths["overview3dPath"],
        "rowCount": int(len(overview3dFrame)),
        "columnCount": int(overview3dFrame.shape[1]),
    },
    {
        "tableKey": "fiturRegresiSdf",
        "title": "Fitur Regresi Lengkap",
        "description": "Seluruh baris fitur hasil ekstraksi SMILES via RDKit.",
        "path": fullTablePaths["featurePath"],
        "rowCount": int(len(featureFrame)),
        "columnCount": int(featureFrame.shape[1]),
    },
    {
        "tableKey": "prediksiDataTest",
        "title": "Prediksi Split Validasi",
        "description": "Prediksi, residual, dan galat pada split validasi resmi OGB.",
        "path": fullTablePaths["predictionPath"],
        "rowCount": int(len(predictionFrame)),
        "columnCount": int(predictionFrame.shape[1]),
    },
    {
        "tableKey": "perbandinganModelValidasi",
        "title": "Perbandingan Model Validasi",
        "description": "Ringkasan performa semua model pada validasi resmi.",
        "path": fullTablePaths["modelComparisonPath"],
        "rowCount": int(len(resultFrame)),
        "columnCount": int(resultFrame.shape[1]),
    },
    {
        "tableKey": "metrikEvaluasiModelTerbaik",
        "title": "Metrik Regresi Lengkap",
        "description": "Seluruh metrik evaluasi untuk model terbaik.",
        "path": fullTablePaths["metricPath"],
        "rowCount": int(len(metricExportFrame)),
        "columnCount": int(metricExportFrame.shape[1]),
    },
    {
        "tableKey": "korelasiFitur",
        "title": "Korelasi Fitur",
        "description": "Korelasi absolut fitur terhadap HOMO-LUMO gap.",
        "path": fullTablePaths["correlationPath"],
        "rowCount": int(len(correlationFrame)),
        "columnCount": int(correlationFrame.shape[1]),
    },
    {
        "tableKey": "fiturTerpenting",
        "title": "Fitur Terpenting",
        "description": "Feature importance jika tersedia dari model terbaik.",
        "path": fullTablePaths["importancePath"],
        "rowCount": int(len(importanceExportFrame)),
        "columnCount": int(importanceExportFrame.shape[1]),
    },
    {
        "tableKey": "riwayatTrainingGin",
        "title": "Riwayat Training GIN",
        "description": "Tetap dibuat untuk kompatibilitas, meskipun kosong pada baseline ini.",
        "path": fullTablePaths["graphHistoryPath"],
        "rowCount": int(len(graphHistoryFrame)),
        "columnCount": int(graphHistoryFrame.shape[1]),
    },
    {
        "tableKey": "ringkasanResidual",
        "title": "Ringkasan Residual",
        "description": "Statistik residual model terbaik pada split validasi.",
        "path": fullTablePaths["residualSummaryPath"],
        "rowCount": int(len(residualSummaryFrame)),
        "columnCount": int(residualSummaryFrame.shape[1]),
    },
    {
        "tableKey": "diagnostikKuantilRegresi",
        "title": "Diagnostik Kuantil Regresi",
        "description": "Perbandingan kuantil aktual dan prediksi.",
        "path": fullTablePaths["quantileDiagnosticPath"],
        "rowCount": int(len(quantileFrame)),
        "columnCount": int(quantileFrame.shape[1]),
    },
]

savedPlotEntries = [
    {
        "plotKey": plotKey,
        "path": pathValue,
    }
    for plotKey, pathValue in plotPathMap.items()
]

savedArtifacts = [
    {
        "fileName": "overviewMolekul3d.csv",
        "path": "output/overviewMolekul3d.csv",
        "description": "Cloud 3D ringan untuk overview molekul berbasis SMILES.",
    },
    {
        "fileName": "fiturRegresiSdf.csv",
        "path": "output/fiturRegresiSdf.csv",
        "description": "Tabel fitur penuh hasil ekstraksi SMILES.",
    },
    {
        "fileName": "prediksiDataTest.csv",
        "path": "output/prediksiDataTest.csv",
        "description": "Prediksi split validasi lengkap dengan residual dan galat.",
    },
    {
        "fileName": "perbandinganModelValidasi.csv",
        "path": "output/perbandinganModelValidasi.csv",
        "description": "Perbandingan semua model pada validasi resmi.",
    },
    {
        "fileName": "metrikEvaluasiModelTerbaik.csv",
        "path": "output/metrikEvaluasiModelTerbaik.csv",
        "description": "Metrik regresi lengkap untuk model terbaik.",
    },
    {
        "fileName": "ringkasanPresentasi.json",
        "path": "output/ringkasanPresentasi.json",
        "description": "Ringkasan eksperimen yang dipakai website presentasi.",
    },
    {
        "fileName": "modelRegresiRadiusGyration.pkl",
        "path": "output/modelRegresiRadiusGyration.pkl",
        "description": "Payload model akhir, tetap memakai nama file lama demi kompatibilitas.",
    },
    {
        "fileName": "notebookShowcase.json",
        "path": "output/notebookShowcase.json",
        "description": "Payload lengkap website untuk merender notebook per cell.",
    },
] + [
    {
        "fileName": Path(plotItem["path"]).name,
        "path": plotItem["path"],
        "description": f"Plot hasil eksperimen untuk {plotItem['plotKey']}.",
    }
    for plotItem in savedPlotEntries
]

showcasePayload = {
    "meta": {
        "title": "Presentasi Interaktif Hasil Training PCQM4Mv2",
        "subtitle": "Versi Kaggle ini memakai download dataset resmi OGB melalui network dan target HOMO-LUMO gap dari SMILES.",
        "notebookPath": "main.ipynb",
        "artifactDirectory": "output",
        "sourceSdf": metadataPayload["sdfPath"],
        "generatedAt": datetime.now(timezone.utc).isoformat(),
    },
    "config": {
        "maxMoleculeCount": int(len(featureFrame)),
        "sampleStride": int(sampleStride),
        "reportInterval": int(reportInterval),
        "targetColumnName": targetColumnName,
        "featureCount": int(len(modelFeatureColumns)),
        "randomState": int(randomState),
    },
    "parsingSummary": {
        "processedMoleculeCount": int(len(featureFrame)),
        "skippedMoleculeCount": int(skippedMoleculeCount),
        "columnCount": int(featureFrame.shape[1]),
    },
    "configurationRows": json.loads(datasetOverviewFrame.to_json(orient="records")),
    "overviewPreview": json.loads(
        featureFrame[
            ["moleculeIndex", "splitName", "atomCount", "heavyAtomCount", "bondCount", targetColumnName]
        ].head(3).to_json(orient="records")
    ),
    "overview3dPreview": json.loads(overview3dFrame.head(10).round(6).to_json(orient="records")),
    "edaSummary": json.loads(edaSummaryFrame.to_json(orient="records")),
    "edaSample": json.loads(edaFrame.head(24).to_json(orient="records")),
    "preprocessSummary": json.loads(preprocessSummaryFrame.to_json(orient="records")),
    "featurePreview": json.loads(featureFrame.head(8).to_json(orient="records")),
    "experimentMoleculeOverview": json.loads(
        featureFrame[
            ["moleculeIndex", "splitName", "moleculeTitle", "atomCount", "heavyAtomCount", "hydrogenCount", "bondCount", targetColumnName]
        ].head(200).to_json(orient="records")
    ),
    "experimentMoleculeCount": int(len(featureFrame)),
    "datasetSummary": {
        "jumlahBaris": int(len(featureFrame)),
        "jumlahKolom": int(featureFrame.shape[1]),
        "nilaiHilangTotal": int(featureFrame.isna().sum().sum()),
        "targetMinimum": round(float(featureFrame[targetColumnName].min()), 6),
        "targetMaksimum": round(float(featureFrame[targetColumnName].max()), 6),
        "targetRataRata": round(float(featureFrame[targetColumnName].mean()), 6),
        "targetMedian": round(float(featureFrame[targetColumnName].median()), 6),
    },
    "statisticsPreview": json.loads(statisticsFullFrame.head(12).round(6).to_json(orient="records")),
    "statisticsFull": json.loads(statisticsFullFrame.round(6).to_json(orient="records")),
    "correlationPreview": json.loads(correlationFrame.head(12).round(6).to_json(orient="records")),
    "correlationFull": json.loads(correlationFrame.round(6).to_json(orient="records")),
    "splitSummary": json.loads(splitFrame.to_json(orient="records")),
    "modelComparison": json.loads(resultFrame.round(6).to_json(orient="records")),
    "bestModel": {
        "modelName": bestModelName,
        "testMae": round(float(testMae), 6),
        "testRmse": round(float(testRmse), 6),
        "testR2": round(float(testR2), 6),
    },
    "metricSummary": json.loads(metricExportFrame.round(6).to_json(orient="records")),
    "predictionPreview": json.loads(predictionFrame.head(10).round(6).to_json(orient="records")),
    "testMoleculePreview": json.loads(testMoleculePreview.round(6).to_json(orient="records")),
    "errorSummary": {
        "galatMinimum": round(float(predictionFrame["galatAbsolut"].min()), 6),
        "galatRataRata": round(float(predictionFrame["galatAbsolut"].mean()), 6),
        "galatMedian": round(float(predictionFrame["galatAbsolut"].median()), 6),
        "galatMaksimum": round(float(predictionFrame["galatAbsolut"].max()), 6),
    },
    "residualSummary": json.loads(residualSummaryFrame.round(6).to_json(orient="records")),
    "quantileDiagnostic": json.loads(quantileFrame.round(6).to_json(orient="records")),
    "graphHistory": json.loads(graphHistoryFrame.round(6).to_json(orient="records")),
    "featureImportance": json.loads(importanceExportFrame.round(6).to_json(orient="records")),
    "savedPlots": savedPlotEntries,
    "presentationSummary": presentationSummaryPayload,
    "fullTablePaths": fullTablePaths,
    "tableCatalog": tableCatalog,
    "savedArtifacts": savedArtifacts,
}

showcasePath.write_text(json.dumps(showcasePayload, ensure_ascii=False, indent=2), encoding="utf-8")

webManifestPayload = {
    "activeArtifactDir": "output",
    "showcasePath": "output/notebookShowcase.json",
    "notebookPath": "main.ipynb",
    "generatedAt": datetime.now(timezone.utc).isoformat(),
}
webManifestPath.write_text(
    json.dumps(webManifestPayload, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

if downloadZipPath.exists():
    downloadZipPath.unlink()
shutil.make_archive(downloadZipPath.with_suffix("").as_posix(), "zip", root_dir=".", base_dir="output")

print(f"Model tersimpan di: {artifactPath}")
print(f"Prediksi evaluasi tersimpan di: {predictionPath}")
print(f"Perbandingan model tersimpan di: {modelComparisonPath}")
print(f"Metrik evaluasi tersimpan di: {metricPath}")
print(f"Ringkasan residual tersimpan di: {residualSummaryPath}")
print(f"Diagnostik kuantil regresi tersimpan di: {quantileDiagnosticPath}")
print(f"Ringkasan presentasi tersimpan di: {presentationSummaryPath}")
print(f"Ringkasan eksperimen tersimpan di: {metadataPath}")
print(f"Showcase website tersimpan di: {showcasePath}")
print(f"Data viewer ringan tersimpan di: {viewerManifestPath}")
if submissionArtifactPath:
    print(f"Submission resmi test-dev tersimpan di: {submissionArtifactPath}")
print(f"Web manifest aktif tersimpan di: {webManifestPath}")
print(f"Paket download Kaggle tersimpan di: {downloadZipPath}")
